# Phenotype ↔ Phenotype Relation-Wise Merge

Merges Phenotype–Phenotype triples from PrimeKG, GPKG, and TARKG;
deduplicates by `(head, relation, tail)`; and saves the result.

## 0. Configuration

In [2]:

import os
import pandas as pd
import numpy as np

BASE_DIR  = '/storage/Arushi/090526_EvoAge/kg_formation/'
DB_DIR     = BASE_DIR + 'data_collection/databases_for_mapping/'
PROC_DIR  = BASE_DIR + 'processed_data/'

# ── Output path ───────────────────────────────────────────────────────────────
OUT_PATH   = BASE_DIR + 'processed_data_relation_wise_merge/generalised/PHENOTYPE_Protein/ALL_PHENOTYPE_Protein.csv'

# ── Required output schema ────────────────────────────────────────────────────

REQUIRED_COLS = [
    'head', 'relation', 'tail',
    'head_type', 'relation_type', 'tail_type',
    'kg_source',
    'head_id_is', 'tail_id_is',
    'head_detail_name', 'tail_detail_name', 'kg_type', 'species'
]

## 1. Load KG Sources

### 1a. PrimeKG

In [10]:
crossbar = pd.read_csv(PROC_DIR + 'crossbar/Phenotype_Protein_Crossbar.csv')
crossbar.columns = crossbar.columns.str.lower()
print(f"crossbar: {len(crossbar):,} rows | columns: {list(crossbar.columns)}")
crossbar.rename(columns={'head_alt_id':'head_detail_name'}, inplace=True, errors='ignore')
crossbar['kg_type'] = 'Generalised'
crossbar['kg_source'] = 'crossbar'
crossbar['species'] = 'Homo species'
crossbar.head(2)

crossbar: 39,720 rows | columns: ['head', 'relation', 'tail', 'head_type', 'tail_type', 'kg_source', 'head_alt_id', 'tail_detail_name', 'head_id_is', 'tail_id_is']


,head,relation,tail,head_type,tail_type,kg_source,head_detail_name,tail_detail_name,head_id_is,tail_id_is,kg_type,species
0,HP:0005357,Phenotype_Protein,Q13315,Phenotype,Protein,crossbar,Defective B cell differentiation,Serine-protein kinase ATM,HPO_ID,Uniprot_ID,Generalised,Homo species
1,HP:0005384,Phenotype_Protein,Q13315,Phenotype,Protein,crossbar,Defective B cell activation,Serine-protein kinase ATM,HPO_ID,Uniprot_ID,Generalised,Homo species


## 2. Check and Fix Duplicate Columns

In [11]:
SOURCE_DFS = [
    ('crossbar', crossbar),
]

for name, df in SOURCE_DFS:
    dupes = df.columns[df.columns.duplicated(keep=False)].unique().tolist()
    if dupes:
        print(f"\n[{name}] duplicate columns: {dupes}")
        for col in dupes:
            for i, (_, s) in enumerate(df.loc[:, df.columns == col].items()):
                print(f"  '{col}' occurrence {i} → sample: {s.dropna().head(3).tolist()}")
    else:
        print(f"[{name}] ✓ no duplicates")

[crossbar] ✓ no duplicates


In [12]:
# # Edit keep='first'/'last' per DataFrame based on inspection above
# crossbar = crossbar.loc[:, ~crossbar.columns.duplicated(keep='first')]
# GPKG_Phenotype_Phenotype    = GPKG_Phenotype_Phenotype.loc[:,    ~GPKG_Phenotype_Phenotype.columns.duplicated(keep='first')]
# TARKG_Phenotype_Phenotype   = TARKG_Phenotype_Phenotype.loc[:,   ~TARKG_Phenotype_Phenotype.columns.duplicated(keep='first')]

# for name, df in SOURCE_DFS:
#     remaining = df.columns[df.columns.duplicated()].tolist()
#     print(f"[{name}] {'✓ clean' if not remaining else '✗ still has dupes: ' + str(remaining)}")

## 3. Align Schemas and Concatenate

In [13]:
CONCAT_COLS = [
    'head', 'relation', 'tail',
    'head_type', 'relation_type', 'tail_type',
    'kg_source', 'head_id_is', 'tail_id_is',
    'head_detail_name', 'tail_detail_name', 'kg_type', 'species'
]

df_list = []
for name, df in SOURCE_DFS:
    tmp = df.copy()
    for col in CONCAT_COLS:
        if col not in tmp.columns:
            tmp[col] = pd.NA
    df_list.append(tmp[CONCAT_COLS])

final_df = pd.concat(df_list, ignore_index=True)
print(f"Combined: {len(final_df):,} rows")
final_df.head(2)

Combined: 39,720 rows


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species
0,HP:0005357,Phenotype_Protein,Q13315,Phenotype,<NA>,Protein,crossbar,HPO_ID,Uniprot_ID,Defective B cell differentiation,Serine-protein kinase ATM,Generalised,Homo species
1,HP:0005384,Phenotype_Protein,Q13315,Phenotype,<NA>,Protein,crossbar,HPO_ID,Uniprot_ID,Defective B cell activation,Serine-protein kinase ATM,Generalised,Homo species


## 4. Standardise Fixed-Value Columns

In [14]:

print("Unique relation:",   set(final_df['relation']))
print("Unique head_type:",  set(final_df['head_type']))
print("Unique tail_type:",  set(final_df['tail_type']))
print("Unique kg_source:",  set(final_df['kg_source']))

Unique relation: {'Phenotype_Protein'}
Unique head_type: {'Phenotype'}
Unique tail_type: {'Protein'}
Unique kg_source: {'crossbar'}


## 5. Deduplicate and Add Schema Columns

Group by `(head, relation, tail)` and collapse `kg_source` with `::` separator.

In [15]:
GROUP_COLS = ['head', 'relation', 'tail']

def merge_sources(x):
    return '::'.join(sorted(set(x.dropna())))

print(f"Before deduplication: {len(final_df):,} rows")
final_df_dedup = final_df.groupby(GROUP_COLS, as_index=False).agg({
    'head_type':        'first',
    'relation_type':    'first',
    'tail_type':        'first',
    'kg_source':         merge_sources,
    'head_id_is':       'first',
    'tail_id_is':       'first',
    'head_detail_name': 'first',
    'tail_detail_name': 'first',
    'kg_type': merge_sources,
    'species': 'first',
    
})

final_df_dedup = final_df_dedup[REQUIRED_COLS]

print(f"After deduplication: {len(final_df_dedup):,} rows")
final_df_dedup.head()

Before deduplication: 39,720 rows
After deduplication: 39,720 rows


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species
0,HP:0000002,Phenotype_Protein,O95786,Phenotype,None,Protein,crossbar,HPO_ID,Uniprot_ID,Abnormality of body height,Antiviral innate immune response receptor RIG-...,generalised,Human
1,HP:0000003,Phenotype_Protein,O15078,Phenotype,None,Protein,crossbar,HPO_ID,Uniprot_ID,Multicystic kidney dysplasia,Centrosomal protein of 290 kDa,generalised,Human
2,HP:0000003,Phenotype_Protein,P48742,Phenotype,None,Protein,crossbar,HPO_ID,Uniprot_ID,Multicystic kidney dysplasia,LIM/homeobox protein Lhx1,generalised,Human
3,HP:0000003,Phenotype_Protein,Q3SYG4,Phenotype,None,Protein,crossbar,HPO_ID,Uniprot_ID,Multicystic kidney dysplasia,Protein PTHB1,generalised,Human
4,HP:0000003,Phenotype_Protein,Q6ZW61,Phenotype,None,Protein,crossbar,HPO_ID,Uniprot_ID,Multicystic kidney dysplasia,Bardet-Biedl syndrome 12 protein,generalised,Human


## 6. QC — NaN Check

In [16]:
def nan_summary(df):
    true_nan   = df.isna().sum()
    string_nan = df.apply(lambda c: c.astype(str).str.upper().eq('NAN').sum())
    return pd.DataFrame({
        'NaN_count':          true_nan,
        "'NAN'_string_count": string_nan,
        'Total_NaN_like':     true_nan + string_nan
    })

nan_summary(final_df_dedup)

,NaN_count,'NAN'_string_count,Total_NaN_like
head,0,0,0
relation,0,0,0
tail,0,0,0
head_type,0,0,0
relation_type,39720,0,39720
tail_type,0,0,0
kg_source,0,0,0
head_id_is,0,0,0
tail_id_is,0,0,0
head_detail_name,0,0,0


## 7. Save Output

In [7]:
import os
os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)
final_df_dedup.to_csv(OUT_PATH, index=False)
print(f"Saved {len(final_df_dedup):,} rows → {OUT_PATH}")

Saved 39,720 rows → /storage/Arushi/090526_EvoAge/kg_formation/processed_data_relation_wise_merge/generalised/PHENOTYPE_Protein/ALL_PHENOTYPE_Protein.csv


In [6]:
# final_df_dedup = pd.read_csv('/storage/Arushi/090526_EvoAge/kg_formation/processed_data_relation_wise_merge/generalised/PHENOTYPE_Protein/ALL_PHENOTYPE_Protein.csv')
# final_df_dedup['kg_type'] = 'Generalised'
# final_df_dedup